In [ ]:
import os
import warnings
import pandas as pd

# Optional geospatial tools (only import what is used)
try:
    import folium
except:
    !pip install folium
try:
    from localtileserver import get_folium_tile_layer, TileClient
except:
    !pip install localtileserver
try:
    import geopandas as gpd
    from shapely.geometry import Point
except:
    !pip install geopandas
    !pip install shapely

# Suppress warnings
warnings.filterwarnings("ignore")

### Input configuration

In [ ]:
data_folderADS = "data/e2e8ac21-0de5-4cbc-ad2e-128cfb028003"
filename = "CITiZAN_All_features_published.csv"
data_path = os.path.join(data_folderADS, filename)

### Load tabular data

In [ ]:
df = pd.read_csv(data_path)
df.head()

### Convert to geospatial format

In [ ]:
# Create Point geometries from longitude/latitude columns
data_geometry = [
    Point(xy) for xy in zip(df["Long"], df["Lat"])
]

# Build GeoDataFrame in WGS84 coordinate system (EPSG:4326)
gdf = gpd.GeoDataFrame(df, geometry=data_geometry, crs="EPSG:4326")

gdf.head()

### Initialize map

In [ ]:
# Center map on first observation
center_lat = gdf.iloc[0].Lat
center_lon = gdf.iloc[0].Long

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=8
)

### Add markers to map

In [ ]:
for _, row in gdf.iterrows():
    folium.Marker(
        location=[row.Lat, row.Long],
        popup=folium.Popup(
            f"<b>{row.Description}</b>",
            max_width=250
        ),
        icon=folium.Icon(
            color="blue",
            icon="info-sign"
        )
    ).add_to(m)

### Display map

In [ ]:
m